In [ ]:
import json
from pathlib import Path

import numpy as np
from PIL import Image

root = Path("../data")
index = json.loads((root / "index.json").read_text())
names = [i["name"] for i in index if i["split"] == "train"]

name = names[0]
rgb = np.asarray(Image.open(root / "frames" / f"{name}_rgb.png").convert("RGB"))
depth = np.load(root / "frames" / f"{name}_depth.npy")
meta = json.loads((root / "frames" / f"{name}.json").read_text())

points = np.load(root / "frames" / f"{name}_lidar.npy")
column, row, z = points[:, 0].astype(int), points[:, 1].astype(int), points[:, 2]
lidar = np.zeros_like(depth)
lidar[row, column] = z


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def block(ins, outs):
    return nn.Sequential(
        nn.Conv2d(ins, outs, 3, padding=1),
        nn.GroupNorm(8, outs),
        nn.ReLU(),
        nn.Conv2d(outs, outs, 3, padding=1),
        nn.GroupNorm(8, outs),
        nn.ReLU(),
    )


class SnowRefiner(nn.Module):
    def __init__(self, use_lidar=True, use_motion=True, use_transformer=True, sensor_dropout=0.3):
        super().__init__()
        self.use_lidar = use_lidar
        self.use_motion = use_motion
        self.use_transformer = use_transformer
        self.sensor_dropout = sensor_dropout

        # base 1 + lidar 2 + warp 2 + rgb 3 = 8 channels
        self.enc1 = block(8, 32)
        self.enc2 = block(32, 64)
        self.enc3 = block(64, 128)
        self.enc4 = block(128, 256)
        self.middle = block(256, 256)

        if use_transformer:
            layer = nn.TransformerEncoderLayer(256, nhead=8, batch_first=True)
            self.transformer = nn.TransformerEncoder(layer, num_layers=4)
            self.position = nn.Parameter(torch.zeros(1, 256, 45, 80))

        self.dec4 = block(256 + 256, 256)
        self.dec3 = block(256 + 128, 128)
        self.dec2 = block(128 + 64, 64)
        self.dec1 = block(64 + 32, 32)
        self.out = nn.Conv2d(32, 2, 1)

        # starts at zero, so the untrained model returns the DA3 depth
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def sensor_on(self, use_sensor, x):
        # 1 keeps the sensor and 0 switches it off, chosen per sample while training
        on = torch.full((len(x), 1, 1, 1), float(use_sensor), device=x.device)
        if self.training:
            on = on * (torch.rand_like(on) >= self.sensor_dropout)
        return on

    def up(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear")
        return torch.cat([x, skip], dim=1)

    def forward(self, rgb, base, lidar, lidar_mask, warped, warp_mask):
        lidar_on = self.sensor_on(self.use_lidar, rgb)
        motion_on = self.sensor_on(self.use_motion, rgb)

        log_base = base.clamp(min=1e-3).log()
        x = torch.cat([
            log_base,
            lidar.log1p() * lidar_on, lidar_mask * lidar_on,
            warped.log1p() * motion_on, warp_mask * motion_on,
            rgb,
        ], dim=1)

        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        e4 = self.enc4(F.max_pool2d(e3, 2))
        m = self.middle(F.max_pool2d(e4, 2))

        if self.use_transformer:
            b, c, h, w = m.shape
            m = m + F.interpolate(self.position, size=(h, w), mode="bilinear")
            m = self.transformer(m.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(b, c, h, w)

        d = self.dec4(self.up(m, e4))
        d = self.dec3(self.up(d, e3))
        d = self.dec2(self.up(d, e2))
        d = self.dec1(self.up(d, e1))

        correction, uncertainty = self.out(d).chunk(2, dim=1)
        return log_base + correction, uncertainty


In [ ]:
def paper_loss(log_depth, uncertainty, exact):
    # sky is 0 in the exact depth, so it's left out
    valid = exact > 0
    error = torch.abs(log_depth[valid] - exact[valid].log())
    return (error * torch.exp(-uncertainty[valid]) + uncertainty[valid]).mean()


def paper_plus_absrel(log_depth, uncertainty, exact):
    # real AbsRel punishes overshooting on nearby pixels
    valid = exact > 0
    predicted = log_depth[valid].float().exp()
    absrel = (torch.abs(predicted - exact[valid]) / exact[valid]).mean()
    return paper_loss(log_depth, uncertainty, exact) + absrel

In [ ]:
from torch.utils.data import DataLoader

# the dataset is in its own file because workers can't load a class from a notebook on Windows
from snow_data import SnowDataset

# fastest settings that fit in the GPU: batch 4 runs out of real memory and gets about 16x slower
train_loader = DataLoader(SnowDataset("train"), batch_size=2, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(SnowDataset("val"), batch_size=1, num_workers=4, pin_memory=True, persistent_workers=True)

In [ ]:
device = "cuda"
model = SnowRefiner().to(device)
scaler = torch.amp.GradScaler()


def train_one_epoch(loss_function, optimizer, schedule=None):
    model.train()
    total = 0
    for batch in train_loader:
        rgb, base, lidar, lidar_mask, warped, warp_mask, exact = [x.to(device) for x in batch]

        with torch.amp.autocast(device):
            log_depth, uncertainty = model(rgb, base, lidar, lidar_mask, warped, warp_mask)
            loss = loss_function(log_depth, uncertainty, exact)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if schedule:
            schedule.step()
        total += loss.item()
    return total / len(train_loader)


def validate():
    model.eval()
    absrel = 0
    log_error = 0
    with torch.no_grad():
        for batch in val_loader:
            rgb, base, lidar, lidar_mask, warped, warp_mask, exact = [x.to(device) for x in batch]

            with torch.amp.autocast(device):
                log_depth, uncertainty = model(rgb, base, lidar, lidar_mask, warped, warp_mask)

            valid = exact > 0
            predicted = log_depth.float().exp()[valid]
            absrel += (torch.abs(predicted - exact[valid]) / exact[valid]).mean().item()
            log_error += torch.abs(predicted.log() - exact[valid].log()).mean().item()
    return absrel / len(val_loader), log_error / len(val_loader)


# stage 1: the paper's loss, which fixes DA3's scale
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
for epoch in range(6):
    train_loss = train_one_epoch(paper_loss, optimizer)
    absrel, log_error = validate()
    print(f"stage 1 epoch {epoch + 1}: train loss {train_loss:.3f}, val AbsRel {absrel:.3f}, log error {log_error:.3f}", flush=True)

# stage 2: add real AbsRel, with a lower learning rate, and keep the best epoch
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6 * len(train_loader))
best = float("inf")
for epoch in range(6):
    train_loss = train_one_epoch(paper_plus_absrel, optimizer, schedule)
    absrel, log_error = validate()
    print(f"stage 2 epoch {epoch + 1}: train loss {train_loss:.3f}, val AbsRel {absrel:.3f}, log error {log_error:.3f}", flush=True)
    if absrel < best:
        best = absrel
        torch.save(model.state_dict(), "refiner_paper_plus_absrel.pt")